In [ ]:
%pip install comet_ml > /dev/null 2>&1
%pip install mido
%pip install torch
%pip install tqdm
%pip install IPython
%pip install pyyaml

from src.dataset_processing import dataset_processing
from src.model import make_transformer_vae, TransformerVAE
from src.training import train, create_experiment

from .load_checkpoint import load_checkpoint

import yaml
import torch
import torch.optim as optim

### Processing the dataset ###
training_set_path = "PUT YOUR TOKENIZED TRAINING SET PATH HERE" # e.g., "datasets/CPRemi/LateRomantic/Training-Set"
validation_set_path = "PUT YOUR TOKENIZED VALIDATION SET PATH HERE" # e.g., "datasets/CPRemi/LateRomantic/Validation-Set"

vectorized_songs, validation_set, field2idx, idx2field, vocab_sizes = dataset_processing(training_set_path, validation_set_path)

### Creating the model and loading checkpoint ###
config_path = "PUT YOUR CONFIG FILE PATH HERE" # e.g., "configs/config1.yaml"

with open(config_path, "r") as f:
    params = yaml.safe_load(f)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
my_model = make_transformer_vae(params, vocab_sizes)
my_model.to(device)

checkpoint_dir = "PUT YOUR CHECKPOINT DIRECTORY PATH HERE" # e.g., "ckpts/100k"

# You may have no saved .pt files in your ckpts directory yet, in which case the model uses randomly initialized weights.

checkpoint = load_checkpoint(my_model, checkpoint_dir, use_best_ckpt=True, device=device) # Set use_best_ckpt to True to load the best checkpoint, or False to load the latest checkpoint

# Alternatively, you can make a model without creating your own config file by specifying the parameters directly in a dictionary. For example:

#params = dict(
#    num_training_iterations=10000,
#    epochs=10,
#    batch_size=8,
#    seq_length=1024,
#    learning_rate=2e-4,
#    d_model=256,
#    num_heads=8,
#    dropout=0.1,
#    num_layers=4,
#   latent_dim=64,
#    attribute_embedding_dim=32,
#    max_beta=0.1,
#    ratio_zero=0.1,
#    ratio_increase=0.1
#)

#model = TransformerVAE(
#    vocab_sizes=vocab_sizes,
#    latent_dim=params["latent_dim"],
#    attribute_embedding_dim=params["attribute_embedding_dim"],
#    block_size=params["seq_length"],
#    d_model=params["d_model"],
#    num_heads=params["num_heads"],
#    dropout=params["dropout"],
#    num_layers=params["num_layers"]
#)

# Furthemore, the above method is the only way to instantiate the decoder_only_model.
# For further information, see decoder_only_model.py

### Training ###

new_checkpoint_dir = "PUT YOUR NEW CHECKPOINT DIRECTORY HERE" # You may continue using checkpoint_dir if you wish

# Optimizer
optimizer = optim.AdamW(my_model.parameters(), lr=params["learning_rate"])

# Scaler for mixed-precision training <- this prevents underflow
scaler = torch.amp.GradScaler("cuda")

# Create a Comet experiment
experiment_name = "PUT YOUR EXPERIMENT'S NAME HERE"
COMET_API_KEY = "PUT YOUR COMET API KEY HERE"
experiment = create_experiment(experiment_name, params, COMET_API_KEY)

# Run train
train(
    params,
    my_model,
    checkpoint,
    new_checkpoint_dir,
    optimizer,
    scaler,
    vectorized_songs,
    validation_set,
    field2idx,
    experiment,
    device
)




